In [5]:
import requests
import os
import zipfile

def fetch_and_unzip(url: str, filename: str, output_dir: str = "/content/"):
    """
    Fetches a file from the given URL, saves it, and extracts it if it is a ZIP.
    """
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)

    # 1. Download the file
    print(f"Fetching file from: {url}")
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()

        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"File successfully saved to: {filepath}")

        # 2. Unzip if it's a zip file
        if zipfile.is_zipfile(filepath):
            print("📦 Detected ZIP file. Extracting...")
            with zipfile.ZipFile(filepath, 'r') as zip_ref:
                zip_ref.extractall(output_dir)
                print(f"Contents extracted to: {output_dir}")
                # List extracted files
                extracted_files = zip_ref.namelist()
                print(f"Extracted {len(extracted_files)} files.")
        else:
            print("The file is not a zip archive, skipping extraction.")

    except Exception as e:
        print(f"An error occurred: {e}")

# Configure the GitHub URL (raw link for direct download)
# Note: Using the direct zip download URL
file_url = "https://github.com/OTRF/Security-Datasets/raw/master/datasets/compound/apt29/day1/apt29_evals_day1_manual.zip"
output_filename = "dataset.zip"

fetch_and_unzip(file_url, output_filename)

Fetching file from: https://github.com/OTRF/Security-Datasets/raw/master/datasets/compound/apt29/day1/apt29_evals_day1_manual.zip
File successfully saved to: /content/dataset.zip
📦 Detected ZIP file. Extracting...
Contents extracted to: /content/
Extracted 1 files.


In [6]:
"""
Stage 1: Parse & Normalize
Streams the NDJSON log file in chunks and produces a normalized Parquet file.
"""
import json, os, re
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
# Updated to the actual extracted filename from the logs
DATA_PATH         = "/content/apt29_evals_day1_manual_2020-05-01225525.json"
NORMALIZED_PARQUET = "/content/data/normalized.parquet"
CHUNK_SIZE        = 50_000
# ──────────────────────────────────────────────────────────────────────────────

def _safe_int(val, default=0):
    try: return int(val)
    except: return default

def _hex_to_int(val):
    try: return int(val, 16) if isinstance(val, str) and val.lower().startswith("0x") else 0
    except: return 0

def _parse_dt(s):
    try: return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")
    except: return None

def normalize_event(ev: dict) -> dict:
    dt = _parse_dt(ev.get("EventTime", ""))
    img = ev.get("Image") or ev.get("SourceImage") or ""
    return {
        "record_number": _safe_int(ev.get("RecordNumber")),
        "event_time": ev.get("EventTime", ""),
        "event_id": _safe_int(ev.get("EventID")),
        "hostname": ev.get("Hostname", ""),
        "image": img,
        "image_base": img.split("\\")[-1].lower() if img else "",
        "process_depth": img.count("\\") if img else 0,
        "account_name": ev.get("AccountName", ""),
        "granted_access": _hex_to_int(ev.get("GrantedAccess", "0x0")),
        "message": ev.get("Message", ""),
        "message_len": len(ev.get("Message", ""))
    }

def parse_stage():
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"Could not find {DATA_PATH}. Please check extraction results.")

    os.makedirs(os.path.dirname(NORMALIZED_PARQUET), exist_ok=True)
    chunks, records, total = [], [], 0

    print(f"📂 Streaming: {DATA_PATH}")
    with open(DATA_PATH, "r", encoding="utf-8", errors="replace") as fh:
        for line in tqdm(fh, desc="Parsing"):
            line = line.strip()
            if not line: continue
            try:
                records.append(normalize_event(json.loads(line)))
                total += 1
                if len(records) >= CHUNK_SIZE:
                    chunks.append(pd.DataFrame(records))
                    records = []
            except: continue

    if records: chunks.append(pd.DataFrame(records))
    df = pd.concat(chunks, ignore_index=True)

    # Feature engineering: One-hot top EIDs
    top_eids = df["event_id"].value_counts().head(10).index.tolist()
    for eid in top_eids:
        df[f"eid_{eid}"] = (df["event_id"] == eid).astype(np.int8)

    df.to_parquet(NORMALIZED_PARQUET, index=False)
    print(f"✅ Saved {len(df):,} events to Parquet.")
    return df

if __name__ == "__main__":
    df = parse_stage()

📂 Streaming: /content/apt29_evals_day1_manual_2020-05-01225525.json


Parsing: 0it [00:00, ?it/s]

✅ Saved 196,081 events to Parquet.


In [7]:
"""
Stage 2: Anomaly Detection
Loads normalized.parquet, scores every event with Isolation Forest,
projects to 2-D via UMAP, and saves anomalies.parquet.
"""
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import umap

# ── Config ────────────────────────────────────────────────────────────────────
NORMALIZED_PARQUET  = "/content/data/normalized.parquet"
ANOMALIES_PARQUET   = "/content/data/anomalies.parquet"
CONTAMINATION       = 0.05          # expected anomaly fraction
UMAP_SAMPLE         = 60_000        # rows sent to UMAP (memory guard)
RANDOM_STATE        = 42
# ──────────────────────────────────────────────────────────────────────────────

def build_feature_matrix(df: pd.DataFrame):
    """
    Returns (X_scaled, feature_cols) for the isolation forest.
    Uses numeric + engineered one-hot columns available in current Stage 1.
    """
    # Adjusted to match the columns present in the latest Stage 1 execution
    base_cols = ["event_id", "process_depth", "message_len", "granted_access"]
    eid_cols = [c for c in df.columns if c.startswith("eid_")]
    feature_cols = base_cols + eid_cols

    X = df[feature_cols].fillna(0).astype(float).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, feature_cols, scaler

def run_isolation_forest(df: pd.DataFrame, X_scaled: np.ndarray,
                          contamination: float = CONTAMINATION) -> pd.DataFrame:
    print("🌲  Training Isolation Forest …")
    iso = IsolationForest(
        contamination=contamination,
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    df = df.copy()
    df["anomaly_label"] = iso.fit_predict(X_scaled)          # -1 = anomaly
    df["anomaly_score"]  = iso.score_samples(X_scaled)       # lower = more anomalous
    df["is_anomaly"]     = (df["anomaly_label" ] == -1).astype(int)

    n = df["is_anomaly"].sum()
    print(f"1🚨  Anomalies detected: {n:,}  ({n / len(df) * 100:.2f}%)")
    return df

def run_umap(X_scaled: np.ndarray, df: pd.DataFrame,
             sample_size: int = UMAP_SAMPLE) -> go.Figure:
    """UMAP 2-D projection of a random sample, coloured by anomaly score."""
    n = min(sample_size, len(df))
    idx = np.random.default_rng(RANDOM_STATE).choice(len(df), n, replace=False)

    print(f"1📐  UMAP on {n:,} samples …")
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        random_state=RANDOM_STATE,
        low_memory=True,
    )
    emb = reducer.fit_transform(X_scaled[idx])

    plot_df = pd.DataFrame({
        "x": emb[:, 0],
        "y": emb[:, 1],
        "is_anomaly"   : df["is_anomaly"].iloc[idx].values,
        "anomaly_score": df["anomaly_score"].iloc[idx].values,
        "event_id"     : df["event_id"].iloc[idx].values,
        "hostname"     : df["hostname"].iloc[idx].values,
        "image_base"   : df["image_base"].iloc[idx].values,
    })

    fig = px.scatter(
        plot_df, x="x", y="y",
        color="anomaly_score",
        color_continuous_scale=["#e94560", "#f5a623", "#0f3460", "#16213e"],
        symbol="is_anomaly",
        hover_data=["event_id", "hostname", "image_base"],
        title="1🗺0  UMAP — Isolation Forest Anomaly Scores",
        template="plotly_dark",
        opacity=0.55,
        height=650,
        labels={"anomaly_score": "IF Score (lower = more anomalous)",
                "is_anomaly": "Anomaly"},
    )
    fig.update_traces(marker_size=3)
    return fig

def anomaly_stage(normalized_path: str = NORMALIZED_PARQUET,
                  anomalies_path: str   = ANOMALIES_PARQUET,
                  contamination: float  = CONTAMINATION) -> pd.DataFrame:
    os.makedirs(os.path.dirname(anomalies_path), exist_ok=True)
    print("1📥  Loading normalized parquet …")
    df = pd.read_parquet(normalized_path)
    X_scaled, feature_cols, _ = build_feature_matrix(df)
    df = run_isolation_forest(df, X_scaled, contamination)
    fig = run_umap(X_scaled, df)
    fig.show()
    anomalies_df = df[df["is_anomaly"] == 1].copy()
    anomalies_df.to_parquet(anomalies_path, index=False)
    print(f"1💾  Anomalies saved → {anomalies_path}")
    return anomalies_df

if __name__ == "__main__":
    anomaly_stage()"

AttributeError: module 'plotly.express' has no attribute 'Figure'

In [ ]:
"""
Stage 3: BERTopic — Topic Modelling on Anomalous Events
Cleans log text, embeds with sentence-transformers, clusters with HDBSCAN,
and visualises topics interactively.
"""
import os
import re
import nltk
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# ── Config ────────────────────────────────────────────────────────────────────
ANOMALIES_PARQUET = "/content/data/anomalies.parquet"
TOPICS_PARQUET    = "/content/data/anomalies_with_topics.parquet"
TOPIC_MODEL_DIR   = "/content/data/bertopic_model"
EMBED_MODEL       = "all-MiniLM-L6-v2"
MIN_CLUSTER_SIZE  = 15
TOP_N_WORDS       = 10
# ──────────────────────────────────────────────────────────────────────────────


# ── Text cleaning ─────────────────────────────────────────────────────────────
_GUID_RE      = re.compile(r"\{[0-9a-fA-F\-]{8,}\}")
_HEX_RE       = re.compile(r"\b0x[0-9a-fA-F]+\b")
_TS_RE        = re.compile(r"\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}(?:\.\d+)?")
_PATH_RE      = re.compile(r"[A-Za-z]:\\(?:[^\s\r\n|,\\]+\\)*([^\s\r\n|,\\]+)")
_NUM_RE       = re.compile(r"\b\d+\b")
_WS_RE        = re.compile(r"\s+")


def _ensure_stopwords():
    try:
        from nltk.corpus import stopwords
        return set(stopwords.words("english"))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        from nltk.corpus import stopwords
        return set(stopwords.words("english"))


STOP_WORDS = _ensure_stopwords()

# Windows / Sysmon noise words that add no semantic value
DOMAIN_NOISE = {
    "rulename", "utctime", "processguid", "processid", "image",
    "targetprocessid", "targetprocessguid", "sourcename", "channel",
    "keywords", "opcodevalue", "severityvalue", "eventreceivedtime",
    "sourcemodulename", "sourcemoduletype", "version", "task",
    "threadid", "recordnumber", "executionprocessid", "providerguid",
    "timestamp", "version", "none", "null", "true", "false",
}


def clean_log_text(text: str) -> str:
    """
    Strip technical noise (GUIDs, hex, paths, timestamps, numbers)
    and return a bag-of-meaningful-words string.
    """
    # Replace paths with just the exe/filename token
    text = _PATH_RE.sub(lambda m: " " + m.group(1).lower() + " ", text)
    text = _GUID_RE.sub(" ", text)
    text = _HEX_RE.sub(" hexval ", text)
    text = _TS_RE.sub(" ", text)
    text = _NUM_RE.sub(" ", text)
    # Keep only alpha
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = _WS_RE.sub(" ", text).strip().lower()

    tokens = [
        w for w in text.split()
        if w not in STOP_WORDS
        and w not in DOMAIN_NOISE
        and len(w) > 2
    ]
    return " ".join(tokens) if tokens else "unknown_event"


# ── BERTopic pipeline ─────────────────────────────────────────────────────────

def build_topic_model() -> BERTopic:
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42,
        low_memory=True,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )
    vectorizer = CountVectorizer(
        stop_words="english",
        min_df=2,
        ngram_range=(1, 2),
        max_features=10_000,
    )
    embedding_model = SentenceTransformer(EMBED_MODEL)

    return BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer,
        top_n_words=TOP_N_WORDS,
        calculate_probabilities=True,
        verbose=True,
    )


def topic_stage(anomalies_path: str = ANOMALIES_PARQUET,
                topics_path: str    = TOPICS_PARQUET,
                model_dir: str      = TOPIC_MODEL_DIR) -> tuple[pd.DataFrame, BERTopic]:
    """
    End-to-end Stage 3 entry point.
    Returns (annotated_df, topic_model).
    """
    os.makedirs(os.path.dirname(topics_path), exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    print("📥  Loading anomalies …")
    df = pd.read_parquet(anomalies_path)
    print(f"    Shape: {df.shape}")

    # ── Prepare corpus ────────────────────────────────────────────────────────
    print("🧹  Cleaning log text …")
    corpus_raw = (
        df["message"].fillna("") + " " +
        df["image_base"].fillna("") + " " +
        df["target_image_base"].fillna("") + " " +
        df["target_object"].fillna("").apply(lambda x: x.split("\\")[-1].lower())
    )
    docs = corpus_raw.apply(clean_log_text).tolist()
    print(f"    Corpus size: {len(docs):,} documents")
    print(f"    Sample doc : {docs[0][:120]}")

    # ── Fit BERTopic ──────────────────────────────────────────────────────────
    print("\n🔬  Fitting BERTopic …")
    topic_model = build_topic_model()
    topics, probs = topic_model.fit_transform(docs)

    df = df.copy()
    df["topic"]       = topics
    df["topic_prob"]  = [float(p.max()) if hasattr(p, "max") else float(p)
                         for p in probs]

    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    print(f"\n✅  Discovered {n_topics} topics  (topic -1 = noise/outliers)")

    # ── Topic info ────────────────────────────────────────────────────────────
    topic_info = topic_model.get_topic_info()
    print("\nTop topics:\n", topic_info.head(12).to_string(index=False))

    # ── Visualisations ────────────────────────────────────────────────────────
    print("\n📊  Generating visualisations …")

    fig_bar = topic_model.visualize_barchart(
        top_n_topics=min(12, n_topics), n_words=8
    )
    fig_bar.update_layout(template="plotly_dark",
                          title="📊 Security Event Topics — Top Keywords")
    fig_bar.show()

    if n_topics >= 2:
        fig_map = topic_model.visualize_topics()
        fig_map.update_layout(template="plotly_dark",
                              title="🗺️ Inter-topic Distance Map")
        fig_map.show()

        fig_heat = topic_model.visualize_heatmap()
        fig_heat.update_layout(template="plotly_dark",
                               title="🔥 Topic Similarity Heatmap")
        fig_heat.show()

    # ── Save ──────────────────────────────────────────────────────────────────
    df.to_parquet(topics_path, index=False)
    topic_model.save(model_dir, serialization="safetensors",
                     save_ctfidf=True, save_embedding_model=EMBED_MODEL)
    print(f"\n💾  Saved annotated parquet → {topics_path}")
    print(f"💾  Saved BERTopic model    → {model_dir}")

    return df, topic_model


def get_topic_summary(topic_model: BERTopic, top_n: int = 15) -> dict:
    """
    Returns {topic_id: 'keyword1, keyword2, …'} for the RAG query builder.
    """
    summary = {}
    for tid in topic_model.get_topic_info()["Topic"].tolist():
        if tid == -1:
            continue
        words = topic_model.get_topic(tid)
        if words:
            summary[tid] = ", ".join([w for w, _ in words[:top_n]])
    return summary


if __name__ == "__main__":
    df, model = topic_stage()
    print("\nTopic keyword summary:")
    for tid, kws in list(get_topic_summary(model).items())[:5]:
        print(f"  Topic {tid}: {kws}")


In [ ]:
"""
Stage 4a: RAG Knowledge Base Builder
Downloads and indexes 5 cybersecurity knowledge sources into ChromaDB.
Sources: MITRE ATT&CK, MITRE D3FEND, MITRE CAR, CISA KEV, SigmaHQ Rules
"""
import os, re, json, subprocess
import requests
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
CHROMA_DIR   = "/content/data/chroma_db"
EMBED_MODEL  = "all-MiniLM-L6-v2"
BATCH_SIZE   = 128
KB_DIR       = "/content/data/kb_raw"
# ──────────────────────────────────────────────────────────────────────────────

_emb_model: SentenceTransformer = None

def _get_embedder():
    global _emb_model
    if _emb_model is None:
        print("🤖  Loading embedding model …")
        _emb_model = SentenceTransformer(EMBED_MODEL)
    return _emb_model


def _get_client():
    os.makedirs(CHROMA_DIR, exist_ok=True)
    return chromadb.PersistentClient(path=CHROMA_DIR)


def _upsert_collection(client: chromadb.Client,
                       name: str,
                       docs: list[str],
                       ids: list[str],
                       metas: list[dict]):
    """Create or replace a ChromaDB collection and batch-embed docs."""
    try:
        client.delete_collection(name)
    except Exception:
        pass
    col = client.create_collection(name)
    embedder = _get_embedder()

    for i in tqdm(range(0, len(docs), BATCH_SIZE), desc=f"  Indexing {name}"):
        bd = docs[i:i+BATCH_SIZE]
        bi = ids[i:i+BATCH_SIZE]
        bm = metas[i:i+BATCH_SIZE]
        emb = embedder.encode(bd, show_progress_bar=False).tolist()
        col.add(documents=bd, ids=bi, embeddings=emb, metadatas=bm)

    print(f"  ✅  {name}: {col.count():,} docs indexed")
    return col


# ── Source 1: MITRE ATT&CK ───────────────────────────────────────────────────

def load_mitre_attack(client):
    print("\n📥  MITRE ATT&CK v14 …")
    url = ("https://raw.githubusercontent.com/mitre/cti/master/"
           "enterprise-attack/enterprise-attack.json")
    data = requests.get(url, timeout=120).json()

    docs, ids, metas = [], [], []
    for obj in data["objects"]:
        if obj.get("type") != "attack-pattern":
            continue
        tech_id, tactic = "", ""
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tech_id = ref.get("external_id", "")
        for phase in obj.get("kill_chain_phases", []):
            tactic = phase.get("phase_name", "")
        name = obj.get("name", "")
        desc = obj.get("description", "")[:1800]
        text = f"ATT&CK {tech_id} — {name}\nTactic: {tactic}\n{desc}"
        uid  = f"attack_{tech_id}_{obj['id'][-6:]}"
        docs.append(text); ids.append(uid)
        metas.append({"source": "mitre_attack", "tech_id": tech_id,
                      "tactic": tactic, "name": name})

    _upsert_collection(client, "mitre_attack", docs, ids, metas)


# ── Source 2: MITRE D3FEND ───────────────────────────────────────────────────

def load_d3fend(client):
    print("\n📥  MITRE D3FEND …")
    url = "https://d3fend.mitre.org/api/technique/all.json"
    try:
        data = requests.get(url, timeout=60).json()
        techniques = data.get("techniques") or data.get("data") or []
    except Exception as e:
        print(f"  ⚠️  D3FEND fetch failed: {e}")
        return

    docs, ids, metas = [], [], []
    for t in techniques:
        tid   = t.get("id") or t.get("d3f_id") or "unknown"
        label = t.get("label") or t.get("name") or ""
        desc  = t.get("definition") or t.get("description") or ""
        text  = f"D3FEND {tid} — {label}\n{desc}"[:2000]
        uid   = f"d3fend_{re.sub(r'[^a-zA-Z0-9_]', '_', tid)}"
        docs.append(text); ids.append(uid)
        metas.append({"source": "d3fend", "d3fend_id": tid, "name": label})

    if docs:
        _upsert_collection(client, "mitre_d3fend", docs, ids, metas)
    else:
        print("  ⚠️  D3FEND returned 0 usable techniques.")


# ── Source 3: MITRE CAR ──────────────────────────────────────────────────────

def load_car(client):
    print("\n📥  MITRE CAR analytics …")
    car_dir = os.path.join(KB_DIR, "car")
    if not os.path.exists(car_dir):
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/mitre-attack/car.git", car_dir],
            check=True, capture_output=True,
        )

    import yaml
    docs, ids, metas = [], [], []
    analytics_dir = os.path.join(car_dir, "analytics")
    if not os.path.exists(analytics_dir):
        print("  ⚠️  CAR analytics directory not found.")
        return

    for fname in os.listdir(analytics_dir):
        if not (fname.endswith(".yaml") or fname.endswith(".yml")):
            continue
        try:
            with open(os.path.join(analytics_dir, fname), "r", encoding="utf-8") as f:
                obj = yaml.safe_load(f)
            title = obj.get("title", fname)
            desc  = obj.get("description", "")
            impl  = " ".join(
                str(i.get("code", ""))
                for i in (obj.get("implementations") or [])
                if isinstance(i, dict)
            )
            text = f"CAR — {title}\n{desc}\nDetection logic: {impl}"[:2000]
            uid  = f"car_{fname.replace('.yaml','').replace('.yml','')[:60]}"
            docs.append(text); ids.append(uid)
            metas.append({"source": "mitre_car", "title": title, "file": fname})
        except Exception:
            continue

    if docs:
        _upsert_collection(client, "mitre_car", docs, ids, metas)


# ── Source 4: CISA KEV ───────────────────────────────────────────────────────

def load_cisa_kev(client):
    print("\n📥  CISA Known Exploited Vulnerabilities …")
    url = ("https://www.cisa.gov/sites/default/files/feeds/"
           "known_exploited_vulnerabilities.json")
    data = requests.get(url, timeout=60).json()

    docs, ids, metas = [], [], []
    for v in data.get("vulnerabilities", []):
        cve  = v.get("cveID", "unknown")
        text = (
            f"CVE: {cve} | Vendor: {v.get('vendorProject','')} | "
            f"Product: {v.get('product','')} | "
            f"Vulnerability: {v.get('vulnerabilityName','')} | "
            f"Required Action: {v.get('requiredAction','')} | "
            f"Due Date: {v.get('dueDate','')} | "
            f"Notes: {v.get('notes','')}"
        )[:2000]
        uid = f"kev_{cve}"
        docs.append(text); ids.append(uid)
        metas.append({"source": "cisa_kev", "cve_id": cve,
                      "product": v.get("product", "")})

    _upsert_collection(client, "cisa_kev", docs, ids, metas)


# ── Source 5: SigmaHQ Rules ──────────────────────────────────────────────────

def load_sigma(client):
    print("\n📥  SigmaHQ detection rules (sparse clone) …")
    sigma_dir = os.path.join(KB_DIR, "sigma")
    if not os.path.exists(sigma_dir):
        subprocess.run(
            ["git", "clone", "--depth=1", "--filter=blob:none", "--sparse",
             "https://github.com/SigmaHQ/sigma.git", sigma_dir],
            check=True, capture_output=True,
        )
        subprocess.run(
            ["git", "sparse-checkout", "set", "rules/"],
            cwd=sigma_dir, check=True, capture_output=True,
        )

    import yaml
    docs, ids, metas, seen = [], [], [], set()
    rules_dir = os.path.join(sigma_dir, "rules")

    for root, _, files in os.walk(rules_dir):
        for fname in files:
            if not fname.endswith(".yml"):
                continue
            uid = f"sigma_{fname[:60]}"
            if uid in seen:
                uid += f"_{len(seen)}"
            seen.add(uid)
            try:
                with open(os.path.join(root, fname), "r",
                          encoding="utf-8", errors="replace") as f:
                    raw = f.read()
                obj   = yaml.safe_load(raw) or {}
                title = obj.get("title", fname)
                desc  = obj.get("description", "")
                tags  = ", ".join(obj.get("tags") or [])
                detect = str(obj.get("detection") or "")
                text = (
                    f"Sigma Rule: {title}\n"
                    f"Tags: {tags}\n"
                    f"Description: {desc}\n"
                    f"Detection: {detect}"
                )[:2000]
                docs.append(text); ids.append(uid)
                metas.append({"source": "sigma", "title": title,
                               "file": fname, "tags": tags})
            except Exception:
                continue

    if docs:
        _upsert_collection(client, "sigma_rules", docs, ids, metas)


# ── Entry point ───────────────────────────────────────────────────────────────

def build_knowledge_base():
    """Download and index all 5 knowledge sources."""
    os.makedirs(KB_DIR, exist_ok=True)
    client = _get_client()

    load_mitre_attack(client)
    load_d3fend(client)
    load_car(client)
    load_cisa_kev(client)
    load_sigma(client)

    print("\n🏁  Knowledge base complete.")
    print(f"    Collections: {[c.name for c in client.list_collections()]}")
    return client


def query_all_collections(client: chromadb.Client,
                          query: str,
                          top_k: int = 5) -> str:
    """
    Semantic search across all indexed collections.
    Returns a single concatenated context string.
    """
    embedder = _get_embedder()
    q_emb = embedder.encode([query]).tolist()
    parts  = []

    for col in client.list_collections():
        try:
            n = min(top_k, col.count())
            if n == 0:
                continue
            res = col.query(query_embeddings=q_emb, n_results=n)
            for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
                src = meta.get("source", col.name).upper()
                parts.append(f"[{src}]\n{doc[:600]}")
        except Exception as e:
            print(f"  ⚠️  {col.name}: {e}")

    return "\n\n---\n\n".join(parts)


if __name__ == "__main__":
    build_knowledge_base()


In [ ]:
!pip install dspy-ai chromadb

"""
Stage 4b: LLM Analysis — DSPy + Ollama
"""
import os, json, subprocess, time
import pandas as pd
try:
    import dspy
except ImportError:
    print("Please run the pip install line above and restart the kernel if needed.")
import chromadb
# ... (rest of the analysis code follows)